# 01. RTN vs GPTQ: Second-Order Hessian Error Compensation

- **Category:** 14_quantisation / 03_llm_ptq_algorithms
- **Lineage & Citations:** Frantar et al. (2022, *GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers*); Hassibi et al. (1993, *Optimal Brain Surgeon*).
- **Purpose:** Formulate post-training quantization as a constrained quadratic minimization over the empirical activation Hessian, updating unquantized trailing weights to cancel accumulated quantization errors.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. Sub Byte Bitpacking, T-MAC LUT, and GGUF K-Quants](../02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. AWQ: Activation-Aware Weight Quantization](02_awq_activation_aware_weight_quantization.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT run unregularized matrix inversion on rank-deficient activation covariances ($N < d_{\text{in}}$ requires $\lambda = 0.01$ damping); do NOT quantize activation-sensitive outlier channels last (Act-Order must prioritize largest $\text{diag}(\mathbf{H})$).

## Mathematical Foundations & Second-Order Hessian Compensation

1. **Second-Order Error Expansion**:
   $$E = \|\mathbf{X}\mathbf{W} - \mathbf{X}\widehat{\mathbf{W}}\|_F^2 \approx \frac{1}{2} \text{Tr}(\Delta \mathbf{W}^T \mathbf{H} \Delta \mathbf{W}), \quad \mathbf{H} = \frac{2}{N} \mathbf{X}^T \mathbf{X}$$
2. **Damped Inverse Hessian**:
   $$\mathbf{H}_{\text{damped}} = \mathbf{H} + \lambda \cdot \text{mean}(\text{diag}(\mathbf{H})) \cdot \mathbf{I}, \quad \mathbf{H}^{-1} = (\mathbf{H}_{\text{damped}})^{-1}$$
3. **GPTQ Trailing Column Compensation**:
   $$\mathbf{e}_c = \mathbf{w}_c - \text{quant}(\mathbf{w}_c), \quad \mathbf{w}_{r, :} \leftarrow \mathbf{w}_{r, :} - \frac{[\mathbf{H}^{-1}]_{r, c}}{[\mathbf{H}^{-1}]_{c, c}} \mathbf{e}_c^T \quad (r > c)$$
4. **Act-Order Permutation**:
   $$\text{perm} = \text{argsort}(\text{diag}(\mathbf{H}), \text{descending}=\text{True}) \implies \text{quantize salient channels first}$$


In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def rtn_quantize(weight: torch.Tensor, qmin: float = -8.0, qmax: float = 7.0) -> torch.Tensor:
    scale = torch.max(torch.abs(weight), dim=0, keepdim=True).values / qmax
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    return torch.clamp(torch.round(weight / scale), qmin, qmax) * scale

def compute_damped_hessian_inv(x: torch.Tensor, damp_factor: float = 0.01) -> tuple[torch.Tensor, torch.Tensor]:
    n_samples, d_in = x.shape[0], x.shape[1]
    H = 2.0 * (x.T @ x) / float(n_samples)
    diag_mean = float(torch.mean(torch.diag(H)))
    damp = damp_factor * diag_mean if diag_mean > 0.0 else 1e-4
    H_damped = H + damp * torch.eye(d_in, dtype=H.dtype, device=H.device)
    return H, torch.inverse(H_damped)

def gptq_quantize(weight: torch.Tensor, h_inv: torch.Tensor, qmin: float = -8.0, qmax: float = 7.0) -> torch.Tensor:
    w = weight.clone()
    d_in = w.shape[0]
    scale = torch.max(torch.abs(weight), dim=0, keepdim=True).values / qmax
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    for c in range(d_in):
        w_c = w[c, :]
        s = scale[0, :]
        q_c = torch.clamp(torch.round(w_c / s), qmin, qmax) * s
        err_c = w_c - q_c
        h_inv_cc = float(h_inv[c, c])
        if h_inv_cc != 0.0 and c + 1 < d_in:
            step = err_c.unsqueeze(0) / h_inv_cc
            w[c + 1 :, :] -= step * h_inv[c + 1 :, c].unsqueeze(1)
        w[c, :] = q_c
    return w

def gptq_with_act_order(x: torch.Tensor, weight: torch.Tensor, damp_factor: float = 0.01) -> torch.Tensor:
    H = 2.0 * (x.T @ x) / float(x.shape[0])
    perm = torch.argsort(torch.diag(H), descending=True)
    inv_perm = torch.argsort(perm)
    _, H_inv_perm = compute_damped_hessian_inv(x[:, perm], damp_factor=damp_factor)
    w_perm_quant = gptq_quantize(weight[perm, :], H_inv_perm)
    return w_perm_quant[inv_perm, :]

In [ ]:
# 1. Linear anchor
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. Hessian and Damped Inverse parity
H, H_inv = compute_damped_hessian_inv(weights["x_calib"], damp_factor=0.01)
assert float(torch.max(torch.abs(H - weights["h_matrix"]))) < 1e-5
assert float(torch.max(torch.abs(H_inv - weights["h_inv"]))) < 1e-5

# 3. RTN Quantization parity
w_rtn = rtn_quantize(weights["w_orig"])
assert float(torch.max(torch.abs(w_rtn - weights["w_rtn"]))) < 1e-5

# 4. Standard GPTQ parity
w_gptq = gptq_quantize(weights["w_orig"], weights["h_inv"])
assert float(torch.max(torch.abs(w_gptq - weights["w_gptq"]))) < 1e-5

# 5. GPTQ with Act-Order parity
w_act = gptq_with_act_order(weights["x_calib"], weights["w_orig"])
assert float(torch.max(torch.abs(w_act - weights["w_act_gptq"]))) < 1e-5
print("All RTN, Hessian inverse, and GPTQ parity assertions verified!")

In [ ]:
# Comparative SQNR Benchmark: RTN vs GPTQ vs GPTQ+Act-Order
x_eval = weights["x_calib"]
y_true = x_eval @ weights["w_orig"]
sig_pwr = float(torch.mean(y_true ** 2))

def eval_sqnr(w_q: torch.Tensor) -> float:
    y_pred = x_eval @ w_q
    mse = float(torch.mean((y_true - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_rtn = eval_sqnr(weights["w_rtn"])
sqnr_gptq = eval_sqnr(weights["w_gptq"])
sqnr_act = eval_sqnr(weights["w_act_gptq"])

print(f"RTN SQNR:           {sqnr_rtn:.2f} dB")
print(f"GPTQ SQNR:          {sqnr_gptq:.2f} dB (gain: +{sqnr_gptq - sqnr_rtn:.2f} dB)")
print(f"GPTQ+ActOrder SQNR: {sqnr_act:.2f} dB (gain: +{sqnr_act - sqnr_rtn:.2f} dB)")

assert sqnr_gptq > sqnr_rtn, "GPTQ must outperform RTN"
assert sqnr_act >= sqnr_gptq, "Act-order must match or outperform standard GPTQ"

In [ ]:
# Trailing Weight Compensation Microbenchmark
d_in, d_out = 1024, 1024
err_mock = torch.randn(1, d_out)
h_col_mock = torch.randn(d_in - 1, 1)
w_trail = torch.randn(d_in - 1, d_out)

t0 = time.perf_counter()
for _ in range(50):
    delta = (err_mock / 1.5) * h_col_mock
    w_trail -= delta
t_rank1 = (time.perf_counter() - t0) / 50 * 1e3
print(f"Rank-1 Trailing Compensation (1024x1024) latency: {t_rank1:.3f} ms")
assert t_rank1 < 10.0, "Rank-1 outer-product update must execute in <10 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Hessian Error Compensation**: GPTQ uses empirical second-order information to update trailing weights, canceling accumulated quantization noise.
2. **Act-Order Saliency**: Sorting by $\text{diag}(\mathbf{H})$ quantizes the most critical activation dimensions first while maximum degrees of freedom remain.
3. **Systems Tractability**: Precomputing $\mathbf{H}^{-1}$ enables $O(d^3)$ total runtime, quantizing 70B parameter models in hours.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_rtn_vs_gptq_hessian_error_compensation.md](../../../markdown/14_quantisation/03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.md)
- **PyTorch 2.x Module:** [01_rtn_vs_gptq_hessian_error_compensation.py](../../../pytorch2/14_quantisation/03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.py)
- **Functional JAX Module:** [01_rtn_vs_gptq_hessian_error_compensation.py](../../../jax/14_quantisation/03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.py)
- **Native Rust Candle Track:** [01_rtn_vs_gptq_hessian_error_compensation.rs](../../../candle/14_quantisation/03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.rs)
- **Hardware Kernel (CPU / WGSL):** [01_rtn_vs_gptq_hessian_error_compensation.rs](../../../kernels/14_quantisation/03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.rs)
- **Golden SafeTensors:** [01_rtn_vs_gptq_hessian_error_compensation.safetensors](../../../weights/14_quantisation/03_llm_ptq_algorithms/01_rtn_vs_gptq_hessian_error_compensation.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Sub Byte Bitpacking, T-MAC LUT, and GGUF K-Quants](../02_pt2e_torchao_and_bitpacking/03_sub_byte_bitpacking_tmac_lut_and_gguf_k_quants.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 02. AWQ: Activation-Aware Weight Quantization](02_awq_activation_aware_weight_quantization.ipynb) &rarr;